# Extract position of R\V Polarstern from CAMS gridded data

In [1]:
import xarray as xr
import datetime as dt
from glob import glob
import pandas as pd
import numpy as np
import pyproj
import matplotlib.pyplot as plt


In [2]:
def extract_point_from_grid(lons, lats, target_lon, target_lat):
    '''
    For a single lat, lon point given by point_coords, function returns the indices of
    the closest point from gridded data output, using pyproj package.
    lons, lats: arrays of lon and lat grid to extract from. must have same length
    target_lon, target_lat: floats of coordinates to extract
    inds_min_dist: tuple of indices corresponding to a grid cell in gridded data
                   with minimum distance to target. coords are given as
                   (j, i).
    '''
    geod = pyproj.Geod(ellps="WGS84")
    _, _, dists = geod.inv(
        np.full(lons.shape, target_lon),
        np.full(lats.shape, target_lat),
        lons,
        lats,
    )
    inds_min_dist = np.unravel_index(np.argmin(dists), lons.shape)
    return inds_min_dist


## Read CAMS aerosol fields and Polarstern coordinates

In [3]:
cams = xr.open_mfdataset("/mnt/data/cams/cams_eac4*.nc")
mosaic = xr.open_dataset("/mnt/data/mosaic/nav/mosaic_coords_2020-04.nc").sel(time=cams.time)

# map CAMS aer categories to WRF-Chem species
cams = cams.rename({"aermr07":"oc_hydrophilic"})
cams = cams.rename({"aermr08":"oc_hydrophobic"})
cams = cams.rename({"aermr09":"bc_hydrophilic"})
cams = cams.rename({"aermr10":"bc_hydrophobic"})
cams = cams.rename({"aermr11":"so4"})



## Perform extraction

In [4]:
window = 3
cams_xx, cams_yy = np.meshgrid(cams.longitude.values, cams.latitude.values)

cams_mosaic = []
for t in range(len(cams.time)):
    lon, lat = mosaic.Longitude.values[t], mosaic.Latitude.values[t]
    j, i = extract_point_from_grid(cams_xx, cams_yy, lon, lat)
    # make index arrays for window**2 nearest points, making sure 0 < i < nx
    nx, ny = len(cams.longitude), len(cams.latitude)
    r = window // 2
    imin, imax = max(0, i - r), min(nx, i + r + 1)
    jmin, jmax = max(0, j - r), min(ny, j + r + 1)
    islice = range(imin, imax)
    jslice = range(jmin, jmax)
    subset = cams.isel(time=t, latitude=jslice, longitude=islice)
    subset = subset.assign_coords({
            'i': ('longitude', [0,1,2]),
            'j': ('latitude', [0,1,2]),
        }).swap_dims({'longitude':'i', 'latitude':'j'})
    cams_mosaic.append(subset)
cams_mosaic = xr.concat(cams_mosaic, dim="time")

/tmp/ipykernel_14859/2336966910.py:21: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  cams_mosaic = xr.concat(cams_mosaic, dim="time")


In [5]:
cams_mosaic.drop_vars(["longitude","latitude"]).to_netcdf("/mnt/data/cams/cams_mosaic.nc")